# Lesson 26 Lab — Mixed-Bit Strategies and Sensitive-Layer Fallback

**Puzzle:** If only a few layers cause most quantization error, should every layer use more bits?

The saved outputs were generated by executing every code cell on the recorded RTX 5090. Run all cells to regenerate the evidence on your own CUDA GPU.

## 0. Predict before running

Write down: (1) the expected direction, (2) the mechanism, (3) the observation that would reverse your prediction, and (4) the evidence level required for the claim.

## 1. Theory — objects and data flow

Mixed-bit design assigns a precision/configuration to each layer or group under a memory, latency, and quality budget.

### Core mechanism

A sensitivity scan replaces one layer at a time and measures downstream change. A simple allocation then spends extra bits on the largest marginal quality benefit per added byte; interactions require re-evaluating the assembled model.

In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "26-mixed-bit-fallback"
device = require_cuda()
torch.manual_seed(2026 + 26)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 2. Connect theory to the experiment

### Engineering trade-off

More bit variants improve the Pareto frontier but fragment kernels, packing, and deployment. Layerwise rankings can change when several layers are quantized together.

### What this code tests

The six-layer CUDA lab ranks INT4 substitutions, gives two layers INT8, computes average bits, and re-runs end to end.

**Experiment:** Quantize a six-layer CUDA MLP one layer at a time, rank sensitivity, then construct a budgeted INT4/INT8 mixed-bit candidate.

**Declared evidence label:** `pytorch-gpu`. Check that the shapes, controlled variables, and units match the theoretical question before executing.

In [2]:
dims=[256]*7; weights=[torch.randn(dims[i+1],dims[i],device=device)*0.05 for i in range(6)]; x=torch.randn(512,256,device=device)
def forward(ws):
    y=x
    for i,w in enumerate(ws): y=y@w.t(); y=torch.nn.functional.gelu(y) if i<5 else y
    return y
ref=forward(weights); q4=[]; q8=[]
for w in weights:
    q4.append(symmetric_quantize(w,bits=4,group_size=64)[2]); q8.append(symmetric_quantize(w,bits=8,group_size=64)[2])
sensitivity=[]
for i in range(6):
    ws=list(weights); ws[i]=q4[i]; sensitivity.append({"layer":i,"rmse":error_metrics(ref,forward(ws))["rmse"]})
fallback={r["layer"] for r in sorted(sensitivity,key=lambda z:z["rmse"],reverse=True)[:2]}; mixed=[q8[i] if i in fallback else q4[i] for i in range(6)]
bits=sum((8 if i in fallback else 4)*weights[i].numel() for i in range(6))/sum(w.numel() for w in weights)
result=base_result(26,"pytorch-gpu"); result.update({"layer_sensitivity":sensitivity,"int8_fallback_layers":sorted(fallback),
    "average_weight_bits":round(bits,3),"assembled_output_error":error_metrics(ref,forward(mixed)),
    "conclusion":"A budgeted mixed-bit candidate spent extra precision on measured sensitive layers and was re-evaluated end to end."})


## 3. Inspect the evidence

Compare the final end-to-end error and estimated storage, not only isolated layer rankings.

### Acceptance and rollback gate

Freeze calibration/evaluation, record isolated sensitivities, budget, chosen fallback layers, final assembled quality, storage, operator coverage, and latency.

In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "assembled_output_error": {
    "cosine": 0.97619838,
    "mae": 0.0019622,
    "max_abs": 0.01424789,
    "rmse": 0.00248394
  },
  "average_weight_bits": 5.333,
  "conclusion": "A budgeted mixed-bit candidate spent extra precision on measured sensitive layers and was re-evaluated end to end.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:46:17+00:00",
  "int8_fallback_layers": [
    0,
    1
  ],
  "layer_sensitivity": [
    {
      "layer": 0,
      "rmse": 0.0013786
    },
    {
      "layer": 1,
      "rmse": 0.00130424
    },
    {
      "layer": 2,
      "rmse": 0.00120898
    },
    {
      "layer": 3,
      "rmse": 0.00125767
    },
    {
      "layer": 4,
      "rmse": 0.00121797
    },
    {
      "layer": 5,
      "rmse": 0.00121779
    }
  ],
  "

## 4. Explain the result

Use sensitivity scans to spend precision where it protects the objective, then re-measure the assembled model.

Relate the measured fields back to the mechanism above. Treat the checked-in result as one hardware/software observation, not a universal ranking. The complete derivation, evidence boundary, and primary references are in [`README.md`](README.md).